## EXPLORATORY DATA ANALYSIS OF EVENTSHIELD365

The main task of this notebook is to perform EDA on the raw data
collected from both project data sources — event data from
**Ticketmaster** and historical weather data from the **Open-Meteo**
open-source API.

This analysis is intended to build a hands-on understanding of the raw
data before transformation — specifically checking data volume per
city, missing or unreliable fields, and any structural inconsistencies
between sources. Documenting these limitations and boundaries here
ensures they are accounted for during Transform and Load, rather than
discovered later inside the pipeline.

**PHASE 1 :   LOADING THE DATA**

In this phase we will load the data first and then after loading before analyzing things for which our approach is to use the python's libraries like **Pandas** , **Numpy** & **JSON**  , using them we will load the data in the note book 

In [1]:
import pandas as pd  
import json 
import numpy as np  
from pathlib import Path


event_files = list(Path("../data/raw/raw_events").glob("*.json"))
print(f"Found {len(event_files)} event files")

all_events_data = {}
for file_path in event_files:
    with open(file_path, "r") as file:
        all_events_data[file_path.name] = json.load(file)

print(all_events_data.keys())



weather_files = list(Path("../data/raw/raw_weather").glob("*.json"))
print(f"Found {len(weather_files)} weather files")

all_weather_data = {}
for file_path in weather_files:
    with open(file_path, "r") as file:
        all_weather_data[file_path.name] = json.load(file)

print(all_weather_data.keys())

Found 5 event files
dict_keys(['events_Chicago_2026-09-21.json', 'events_LasVegas_2026-09-21.json', 'events_LosAngeles_2026-09-21.json', 'events_Miami_2026-09-21.json', 'events_NewYork_2026-09-21.json'])
Found 5 weather files
dict_keys(['weather_Chicago_2016_2025.json', 'weather_LasVegas_2016_2025.json', 'weather_LosAngeles_2016_2025.json', 'weather_Miami_2016_2025.json', 'weather_NewYork_2016_2025.json'])


The loading part of the data was successful as we can observe in the output cell. Now the next task is to check the events count per city.

**PHASE 2: CHECKING THE INSIGHTS**

1) THIS ONE IS FOR THE EVENTS , WE ARE CHECKING THE EVENT COUNT PETR CITY IN ORDER TO KNOW IF THERE ARE ANY CITIES WITH ABNORMAL VALUES , EITHER TOO HIGH OR NO VALUES FOR A LONGER TIME PERIOD

In [2]:
sample_filename = list(all_events_data.keys())[0]
sample_events = all_events_data[sample_filename]

events_list = sample_events["events"]   
print(len(events_list))

200


From This we can observe that each city has 200 events , which means there are no outliers in this field of the event data , now we can work with this , but before that we will have a look at the weather data as well.

In [3]:
# Look at one weather file's keys to understand its shape
sample_weather = list(all_weather_data.values())[0]
print(sample_weather.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


Now we will check if windgusts_10m_max actually returned real data

In [4]:
sample_daily = sample_weather["daily"]
print(sample_daily.keys())
print(sample_daily["windgusts_10m_max"][:10])  # first 10 values

dict_keys(['time', 'precipitation_sum', 'apparent_temperature_max', 'weathercode', 'windgusts_10m_max'])
[49.3, 51.5, 43.6, 48.6, 38.5, 33.8, 31.3, 23.8, 51.5, 59.0]


AND YES IT DOES RETURN THE REAL VALUES HENCE THE DATA IS PERFECT TILL NOW 

**PHASE 3 : DATA DESCRIPTION**

Now in this Phase we will be checking the basic informations about the data , before this phase we were just taking a look at the loaded data that if it is correctly formated , now when it is confirmed , we can now check the shape , size and other information about the data .

In [5]:
# THIS IS FOR THE WEATHER DATA 

import pandas as pd

filenames = list(all_weather_data.keys())
first_file = filenames[0]

weather_json = all_weather_data[first_file]
daily_data = weather_json["daily"]

weather_df = pd.DataFrame(daily_data)

print("THE NUMBER OF ROWS AND COLUMNS ARE : " ,weather_df.shape)
weather_df.head()


THE NUMBER OF ROWS AND COLUMNS ARE :  (3653, 5)


,time,precipitation_sum,apparent_temperature_max,weathercode,windgusts_10m_max
0,2016-01-01,0.0,-7.3,3,49.3
1,2016-01-02,0.0,-5.4,1,51.5
2,2016-01-03,0.0,-7.0,3,43.6
3,2016-01-04,0.0,-6.8,71,48.6
4,2016-01-05,0.0,-4.8,3,38.5


In [6]:
# THIS IS FOR THE EVENT DATA 

filenames = list(all_events_data.keys())
first_file = filenames[0]

events_json = all_events_data[first_file]
events_list = events_json["events"]   

names = []
dates = []
segments = []
prices = []

for event in events_list:
    names.append(event.get("name"))
    dates.append(event["dates"]["start"]["localDate"])
    
    if event.get("classifications"):
        segments.append(event["classifications"][0]["segment"]["name"])
    else:
        segments.append(None)
    
    if event.get("priceRanges"):
        prices.append(event["priceRanges"][0]["min"])
    else:
        prices.append(None)

events_df = pd.DataFrame({
    "name": names,
    "date": dates,
    "segment": segments,
    "min_price": prices
})

print(events_df.shape)
events_df.head()

(200, 4)


,name,date,segment,min_price
0,Chicago Bulls vs. Boston Celtics,2026-12-30,Sports,None
1,Chicago Bulls vs. Boston Celtics,2027-03-09,Sports,None
2,Chicago Blackhawks vs. Boston Bruins,2027-01-01,Sports,None
3,Chicago Bulls vs. New York Knicks,2026-10-28,Sports,None
4,Chicago Bulls vs. Los Angeles Lakers,2027-01-02,Sports,None


In [7]:
segments_found = []
for event in events_list:
    segment = event.get("classifications", [{}])[0].get("segment", {}).get("name")
    segments_found.append(segment)

from collections import Counter
print(Counter(segments_found))

Counter({'Sports': 111, 'Music': 52, 'Arts & Theatre': 34, 'Miscellaneous': 2, 'Film': 1})


In [8]:
from collections import Counter

for filename, content in all_events_data.items():
    events_list = content["events"]
    
    segments_found = []
    for event in events_list:
        segment = event.get("classifications", [{}])[0].get("segment", {}).get("name")
        segments_found.append(segment)
    
    print(filename)
    print(Counter(segments_found))
    print("---")

events_Chicago_2026-09-21.json
Counter({'Sports': 111, 'Music': 52, 'Arts & Theatre': 34, 'Miscellaneous': 2, 'Film': 1})
---
events_LasVegas_2026-09-21.json
Counter({'Music': 100, 'Sports': 74, 'Arts & Theatre': 26})
---
events_LosAngeles_2026-09-21.json
Counter({'Sports': 162, 'Arts & Theatre': 35, 'Music': 3})
---
events_Miami_2026-09-21.json
Counter({'Sports': 155, 'Music': 38, 'Miscellaneous': 4, 'Arts & Theatre': 3})
---
events_NewYork_2026-09-21.json
Counter({'Sports': 124, 'Arts & Theatre': 75, 'Music': 1})
---


From the above we loop through all 5 loaded files in all_events_data  and for each one:

Pulls out its events list


Counts the segments, same logic as your single-city check


Prints the filename and its segment breakdown, then a separator line before moving to the next city

**PHASE 4 : MISSING DATA**

In this phase we will be checking data from both the sources that if they have any missing value , any null values or any outliers

In [9]:
sample_filename = list(all_events_data.keys())[0]
events_list = all_events_data[sample_filename]["events"]

missing_date = sum(1 for e in events_list if not e.get("dates", {}).get("start", {}).get("localDate"))
missing_venue = sum(1 for e in events_list if not e.get("_embedded", {}).get("venues"))

print(f"Missing date: {missing_date}")
print(f"Missing venue: {missing_venue}")

Missing date: 0
Missing venue: 0


From the above analysis we can clearly notice that there are 0 missing date and venue data in the event data by the ticketmaster

Now we will have a look at the weather data , if it is having any potential barriers for us .


In [14]:
sample_filename = list(all_weather_data.keys())[0]
sample_weather = all_weather_data[sample_filename]

daily = sample_weather["daily"]
print(daily.keys())

wind_gusts = daily["windgusts_10m_max"]
print(wind_gusts[:10])

none_count = sum(1 for v in wind_gusts if v is None)
print(f"Missing values: {none_count} out of {len(wind_gusts)}")

dict_keys(['time', 'precipitation_sum', 'apparent_temperature_max', 'weathercode', 'windgusts_10m_max'])
[49.3, 51.5, 43.6, 48.6, 38.5, 33.8, 31.3, 23.8, 51.5, 59.0]
Missing values: 0 out of 3653


Now ,again from the code snippet's output its clear that the data is clean  i.e., it has 0 missing values in both the weather data as well as the event data


**PHASE 5: DATE SPREAD CHECK**

In this phase we will be checking that the events are actually spread across an even spread of 6-12 months windows , rather than been bunched up oddly .

In [11]:
from datetime import datetime

for filename, content in all_events_data.items():
    events_list = content["events"]
    event_dates = []

    for event in events_list:
        date_time = event.get("dates", {}).get("start", {}).get("dateTime")
        if date_time:
            date = datetime.fromisoformat(date_time.replace("Z", "+00:00"))
            event_dates.append(date)

    print(filename)

    if event_dates:
        print("Number of events:", len(events_list))
        print("Earliest event:", min(event_dates))
        print("Latest event:", max(event_dates))
    else:
        print("No event dates found")

    print("---")

events_Chicago_2026-09-21.json
Number of events: 200
Earliest event: 2026-09-22 00:00:00+00:00
Latest event: 2027-06-26 21:30:00+00:00
---
events_LasVegas_2026-09-21.json
Number of events: 200
Earliest event: 2026-09-21 21:00:00+00:00
Latest event: 2027-09-19 03:00:00+00:00
---
events_LosAngeles_2026-09-21.json
Number of events: 200
Earliest event: 2026-09-23 02:10:00+00:00
Latest event: 2027-04-12 00:30:00+00:00
---
events_Miami_2026-09-21.json
Number of events: 200
Earliest event: 2026-09-25 23:10:00+00:00
Latest event: 2027-05-02 00:00:00+00:00
---
events_NewYork_2026-09-21.json
Number of events: 200
Earliest event: 2026-09-21 23:00:00+00:00
Latest event: 2027-04-11 22:00:00+00:00
---


**PHASE 6 : VENUE NAME INSPECTION**



In this phase , we  want to see the actual venue names before writing any classification code — this tells us how many venues we're actually dealing with, whether the names are recognizable/googleable, and whether we'll need a simple manual dictionary lookup (small number of venues) or something more elaborate (if there turn out to be hundreds of obscure ones). EDA is where we gather that information.

In [ ]:
from collections import defaultdict
venues_by_city = defaultdict(set)
for filename, content in all_events_data.items():
    for event in content["events"]:
        venues = event.get("_embedded", {}).get("venues", [])
        if venues:
            venue = venues[0]
            venue_name = venue.get("name")
            city_name = venue.get("city", {}).get("name")
            if venue_name and city_name:
                venues_by_city[city_name].add(venue_name)
for city, venues in sorted(venues_by_city.items()):
    print(f"\n{city}")
    print("-" * len(city))
    for venue in sorted(venues):
        print(venue)


Chicago
-------
Aragon Ballroom
Rate Field
Soldier Field
TAO Chicago
The Chicago Theatre
The Salt Shed Indoors (Shed)
United Center
Wrigley Field

Las Vegas
---------
Allegiant Stadium
Club Paris Hospitality
East Harmon Zone
FORMULA 1 HEINEKEN LAS VEGAS GRAND PRIX
Flamingo Zone
Grand Prix Plaza Zone
Koval Zone by Heineken
Sphere
T-Mobile Arena
T-Mobile Zone at Sphere
West Harmon Zone

Los Angeles
-----------
BMO Stadium
Crypto.com Arena
Hollywood Pantages Theatre
Los Angeles Memorial Coliseum
Pauley Pavilion-UCLA
UNIQLO Field at Dodger Stadium

Miami
-----
FPL Solar Amphitheater at Bayfront Park
Hard Rock Stadium
James L Knight Center
Kaseya Center
Knight Concert Hall-Adrienne Arsht PAC
LoanDepot Park
Nu Stadium

New York
--------
Beacon Theatre
Madison Square Garden
Richard Rodgers Theatre


Venue inspection: 36 unique venues were found across 5 cities. Most venues can be manually classified as Indoor or Outdoor from their names, while a small number of ambiguous/zone-based venues require verification. Therefore, a manually maintained venue classification dictionary is practical for the Transform step.

**PHASE 7 : WEATHER SANITY CHECK**
In this phase we are having a spot-check for few actual vaklues across cities , like- Does Las Vegas's precipitation_sum genuinely look lower on average than Miami's? Does weathercode only contain valid-looking numbers, not something obviously broken like negative values or absurd outliers? This is less about missing data and more about "does this data make real-world sense."

In [ ]:
print(all_weather_data.keys())
sample_weather = list(all_weather_data.values())[0]
print(sample_weather.keys())



dict_keys(['weather_Chicago_2016_2025.json', 'weather_LasVegas_2016_2025.json', 'weather_LosAngeles_2016_2025.json', 'weather_Miami_2016_2025.json', 'weather_NewYork_2016_2025.json'])
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


In [ ]:
for filename, content in all_weather_data.items():
    city = filename.replace("weather_", "").replace("_2016_2025.json", "")
    daily = content["daily"]
    precipitation = daily["precipitation_sum"]
    weathercodes = daily["weathercode"]
    print(f"\n{city}")
    print("Average precipitation:", sum(precipitation) / len(precipitation))
    print("Minimum precipitation:", min(precipitation))
    print("Maximum precipitation:", max(precipitation))
    print("Weather codes:", sorted(set(weathercodes)))


Chicago
Average precipitation: 3.133698330139611
Minimum precipitation: 0.0
Maximum precipitation: 137.8
Weather codes: [0, 1, 2, 3, 51, 53, 55, 61, 63, 65, 71, 73, 75]

LasVegas
Average precipitation: 0.3548316452231043
Minimum precipitation: 0.0
Maximum precipitation: 37.8
Weather codes: [0, 1, 2, 3, 51, 53, 55, 61, 63, 65, 71, 73, 75]

LosAngeles
Average precipitation: 1.240487270736381
Minimum precipitation: 0.0
Maximum precipitation: 91.9
Weather codes: [0, 1, 2, 3, 51, 53, 55, 61, 63, 65, 71, 73]

Miami
Average precipitation: 4.229947987955105
Minimum precipitation: 0.0
Maximum precipitation: 143.9
Weather codes: [0, 1, 2, 3, 51, 53, 55, 61, 63, 65]

NewYork
Average precipitation: 3.5920613194634545
Minimum precipitation: 0.0
Maximum precipitation: 187.2
Weather codes: [0, 1, 2, 3, 51, 53, 55, 61, 63, 65, 71, 73, 75]
